# V6.0: Bottleneck-Only SeqCA Experiment

In [ ]:
# ===== Setup =====
from google.colab import drive
drive.mount('/content/drive')

!nvidia-smi 2>/dev/null || echo 'No GPU'

!pip install -q --cache-dir=/content/drive/MyDrive/pip_cache \
    mamba-ssm causal-conv1d einops \
    transformers nibabel pyyaml tqdm scipy

import os, subprocess, zipfile, time, shutil, glob
REPO_DIR = '/content/TextMamba3D'
DRIVE_BASE = '/content/drive/MyDrive/TextMamba3D'
DRIVE_CKPT = os.path.join(DRIVE_BASE, 'checkpoints')
os.makedirs(DRIVE_CKPT, exist_ok=True)

git_dir = os.path.join(REPO_DIR, '.git')
if os.path.isdir(REPO_DIR) and not os.path.isdir(git_dir):
    shutil.rmtree(REPO_DIR)
if os.path.isdir(git_dir):
    os.chdir(REPO_DIR)
    subprocess.run(['git', 'pull'], check=True)
else:
    for attempt in range(1, 4):
        ret = subprocess.run(
            ['git', 'clone', '--depth', '1',
             'https://github.com/PlutoLei/TextMamba3D.git', REPO_DIR],
            capture_output=True, text=True)
        if ret.returncode == 0:
            break
        print(f'Clone attempt {attempt} failed')
        if os.path.isdir(REPO_DIR):
            shutil.rmtree(REPO_DIR)
        time.sleep(5 * attempt)
    else:
        raise RuntimeError('Clone failed')
    os.chdir(REPO_DIR)

DATA_DIR = './data/BraTS2020/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData'
if not os.path.exists(DATA_DIR):
    os.makedirs(os.path.dirname(DATA_DIR), exist_ok=True)
    with zipfile.ZipFile(f'{DRIVE_BASE}/TextBraTS_data.zip', 'r') as zf:
        zf.extractall(os.path.dirname(DATA_DIR))
ET_CACHE = f'{DRIVE_BASE}/et_enriched.zip'
if os.path.exists(ET_CACHE):
    with zipfile.ZipFile(ET_CACHE, 'r') as zf:
        zf.extractall(DATA_DIR)
print(f'Data: {len([d for d in os.listdir(DATA_DIR) if d.startswith("BraTS")])} cases')

def sync_and_tag(tag):
    local_ckpt = os.path.join(REPO_DIR, 'checkpoints')
    if not os.path.exists(local_ckpt):
        return
    for f in glob.glob(os.path.join(local_ckpt, '*.pth')):
        shutil.copy2(f, os.path.join(DRIVE_CKPT, os.path.basename(f)))
    best = os.path.join(local_ckpt, 'best.pth')
    if os.path.exists(best):
        shutil.copy2(best, os.path.join(DRIVE_CKPT, f'best_{tag}.pth'))
        print(f'Tagged: best_{tag}.pth')
    print(f'Synced to {DRIVE_CKPT}')

print('Setup complete')

## Patch: Bottleneck-Only SeqCA

TextBraTS 论文只在 bottleneck (最深层) 做 SeqCA 融合，获得 ET +2.3%。
我们的 V5.0 在 stage 1,2,3 多尺度融合，文本对 ET 贡献几乎为零 (+0.01%)。

假设：多尺度融合稀释了文本的语义信息。Bottleneck-only 能让文本聚焦在最有语义意义的层。

In [ ]:
# Patch textmamba3d.py: add bottleneck-only fusion mode
import os
os.chdir(REPO_DIR)

# Read current file
with open('models/textmamba3d.py', 'r') as f:
    code = f.read()

# Add fusion_mode parameter and bottleneck-only logic
# Only patch if not already patched
if 'fusion_mode' not in code:
    # 1. Add fusion_mode to __init__
    code = code.replace(
        "fusion_type: str = \"seqca\",",
        "fusion_type: str = \"seqca\",\n"
        "        fusion_mode: str = \"multi_scale\",  # \"multi_scale\" | \"bottleneck_only\""
    )

    # 2. Store fusion_mode
    code = code.replace(
        "self.multi_scale_attn = fusion_cls(",
        "self.fusion_mode = fusion_mode\n"
        "        self.multi_scale_attn = fusion_cls("
    )

    # 3. Modify forward to support bottleneck-only
    old_fuse = '''            fused = self.multi_scale_attn(
                img_features[1:], text_features, attention_mask
            )
            # V4.6 Direction B: gate text contribution per scale
            if self.text_gate is not None:
                fused = self.text_gate(img_features[1:], fused)
            decoder_features = [img_features[0]] + fused'''

    new_fuse = '''            if self.fusion_mode == "bottleneck_only":
                # Only fuse at deepest stage (last element), pass others through
                shallow = img_features[1:-1]  # stages 1,2 — no fusion
                deep = [img_features[-1]]     # stage 3 — fuse with text
                fused_deep = self.multi_scale_attn(deep, text_features, attention_mask)
                if self.text_gate is not None:
                    fused_deep = self.text_gate(deep, fused_deep)
                fused = list(shallow) + list(fused_deep)
            else:
                fused = self.multi_scale_attn(
                    img_features[1:], text_features, attention_mask
                )
                if self.text_gate is not None:
                    fused = self.text_gate(img_features[1:], fused)
            decoder_features = [img_features[0]] + fused'''

    code = code.replace(old_fuse, new_fuse)

    # 4. Fix MultiScaleSeqCA to handle single-stage input
    with open('models/textmamba3d.py', 'w') as f:
        f.write(code)
    print('Patched textmamba3d.py: added fusion_mode parameter')

    # 5. Also patch train.py to read fusion_mode from config
    with open('train.py', 'r') as f:
        train_code = f.read()
    if 'fusion_mode' not in train_code:
        train_code = train_code.replace(
            "fusion_type=model_cfg.get('fusion_type', 'seqca'),",
            "fusion_type=model_cfg.get('fusion_type', 'seqca'),\n"
            "        fusion_mode=model_cfg.get('fusion_mode', 'multi_scale'),"
        )
        with open('train.py', 'w') as f:
            f.write(train_code)
        print('Patched train.py: reads fusion_mode from config')

    # 6. Patch evaluate_full.py
    with open('evaluate_full.py', 'r') as f:
        eval_code = f.read()
    if 'fusion_mode' not in eval_code:
        eval_code = eval_code.replace(
            "fusion_type=model_cfg.get('fusion_type', 'seqca'),",
            "fusion_type=model_cfg.get('fusion_type', 'seqca'),\n"
            "        fusion_mode=model_cfg.get('fusion_mode', 'multi_scale'),"
        )
        with open('evaluate_full.py', 'w') as f:
            f.write(eval_code)
        print('Patched evaluate_full.py: reads fusion_mode from config')
else:
    print('Already patched')

# Verify
import importlib, sys
if 'models.textmamba3d' in sys.modules:
    del sys.modules['models.textmamba3d']
print('Patch applied successfully')

In [ ]:
# Generate bottleneck-only config
import yaml, os
os.chdir(REPO_DIR)

with open('configs/archive/textbrats_a100_v5.yaml') as f:
    base = yaml.safe_load(f)

base['model']['fusion_mode'] = 'bottleneck_only'
base['training']['lr'] = 0.0001
base['training']['epochs'] = 200
base['training']['patience'] = 30
base['training']['gradient_checkpointing'] = False
base['experiment'] = {
    'name': 'V6.0_bottleneck_seqca',
    'description': 'Bottleneck-only SeqCA fusion (TextBraTS design) + lr=1e-4'
}

os.makedirs('configs/autoresearch', exist_ok=True)
with open('configs/autoresearch/V6.0_bottleneck_seqca.yaml', 'w') as f:
    yaml.dump(base, f, default_flow_style=False, sort_keys=False)
print('Config saved: configs/autoresearch/V6.0_bottleneck_seqca.yaml')
print(f"  fusion_mode: {base['model']['fusion_mode']}")
print(f"  fusion_type: {base['model'].get('fusion_type', 'seqca')}")
print(f"  lr: {base['training']['lr']}")
print(f"  epochs: {base['training']['epochs']}")

## Smoke Test

In [ ]:
# Quick smoke test: 2 samples, 1 epoch
import os
os.chdir(REPO_DIR)

!python -u train.py \
    --config configs/autoresearch/V6.0_bottleneck_seqca.yaml \
    --max-samples 2 \
    --max-epochs 1 \
    --no-text-ratio 0.0 \
    --grad-accum 1

## V6.0 Training: Bottleneck-Only SeqCA (200 epochs, from scratch)

核心假设：只在 bottleneck 融合文本，让 SeqCA 聚焦于语义最浓缩的层。
对标 TextBraTS 论文设计。从头训练，不 fine-tune。

In [ ]:
# V6.0 Training
import os, glob
os.chdir(REPO_DIR)

for f in glob.glob(os.path.join(REPO_DIR, 'checkpoints', '*.pth')):
    os.remove(f)
print('Cleaned checkpoints')

!python -u train.py \
    --config configs/autoresearch/V6.0_bottleneck_seqca.yaml \
    --no-text-ratio 0.15 \
    --grad-accum 2

sync_and_tag('V6.0')
print('V6.0 training complete!')

## V6.0 Evaluation

只跑 3 个有价值的组合:
1. text + TTA (主力配置)
2. text only (快速推理)
3. notext + TTA + PP (文本不可用时的 fallback)

In [ ]:
# V6.0 Eval: 3 key configs
import subprocess, re, os
os.chdir(REPO_DIR)

ckpt = os.path.join(DRIVE_CKPT, 'best_V6.0.pth')
if not os.path.exists(ckpt):
    ckpt = os.path.join(REPO_DIR, 'checkpoints', 'best.pth')
if not os.path.exists(ckpt):
    ckpt = os.path.join(DRIVE_CKPT, 'last.pth')
assert os.path.exists(ckpt), f'No checkpoint: {ckpt}'
print(f'Using: {ckpt}')

CONFIG = 'configs/autoresearch/V6.0_bottleneck_seqca.yaml'
BASELINE = {'ET': 0.7910, 'TC': 0.8560, 'WT': 0.8967, 'Mean': 0.8479}

eval_configs = [
    ('text+TTA',         ['--use-text', '--tta']),
    ('text',             ['--use-text']),
    ('notext+TTA+PP',   ['--no-text', '--tta', '--advanced-pp']),
]

results = {}
for name, flags in eval_configs:
    print(f'\n{"="*60}')
    print(f'{name}')
    print(f'{"="*60}')
    cmd = ['python', '-u', 'evaluate_full.py',
           '--config', CONFIG,
           '--checkpoint', ckpt,
           '--split', 'test',
           '--overlap', '0.5'] + flags
    ret = subprocess.run(cmd, capture_output=True, text=True)

    metrics = {}
    for line in ret.stdout.split('\n'):
        for key in ['dice_ET', 'dice_TC', 'dice_WT', 'dice_mean']:
            m = re.search(rf'{key}: ([\d.]+) \+/- ([\d.]+)', line)
            if m:
                metrics[key] = float(m.group(1))
                print(f'  {key}: {m.group(1)} +/- {m.group(2)}')

    if ret.returncode != 0:
        print(f'ERROR: {ret.stderr[-300:]}')

    results[name] = metrics

# Summary comparison
print(f'\n{"="*60}')
print(f'V6.0 (Bottleneck SeqCA) vs V5.0 (Multi-Scale SeqCA)')
print(f'{"="*60}')
print(f'{"Config":<20} {"ET":>8} {"TC":>8} {"WT":>8} {"Mean":>8} {"vs V5.0":>10}')
print(f'{"-"*66}')
print(f'{"V5.0 baseline":<20} {BASELINE["ET"]:>7.4f} {BASELINE["TC"]:>7.4f} {BASELINE["WT"]:>7.4f} {BASELINE["Mean"]:>7.4f}')
for name, m in results.items():
    mean = m.get('dice_mean', 0)
    delta = mean - BASELINE['Mean']
    sign = '+' if delta >= 0 else ''
    print(f'{name:<20} {m.get("dice_ET",0):>7.4f} {m.get("dice_TC",0):>7.4f} {m.get("dice_WT",0):>7.4f} {mean:>7.4f} {sign}{delta:>8.4f}')

# Text guidance delta
if 'text+TTA' in results:
    notext_key = 'notext+TTA+PP'
    if notext_key in results:
        t = results['text+TTA']
        n = results[notext_key]
        print(f'\nText guidance delta (text+TTA vs notext+TTA+PP):')
        for k, short in [('dice_ET','ET'), ('dice_TC','TC'), ('dice_WT','WT'), ('dice_mean','Mean')]:
            d = t.get(k,0) - n.get(k,0)
            print(f'  {short}: {d:+.4f}')